# TP6 — Sac de mots, TF-IDF et mini moteur de recherche
### EMSI — Traitement Automatique du Langage Naturel (TALN)
**Compte rendu exécuté — Anwar EZZIRANI · 5IIR · groupe G4 · site Roudani**

Ce notebook suit l'énoncé sur les 100 avis du corpus fil rouge. Il contient les calculs, les sorties, six requêtes commentées et une évaluation manuelle P@3 du moteur.


## Mode d'emploi
1. **Renseignez votre identification**. 2. Exécutez les cellules **dans l'ordre**.
3. Complétez les lignes `...   # ← À COMPLÉTER` ; les cellules « ✅ Vérification » affichent ✅ ou ❌ sans arrêter le notebook.
4. Répondez aux **📝 Interprétations** en citant vos résultats. 5. Enregistrez sous le nom affiché, puis déposez le notebook.

## Partie 0 — Préparation (0:10 – 0:15)
### 0.1 Identification

In [ ]:
NUM_TP = 6
NOM = "EZZIRANI"
PRENOM = "Anwar"
CLASSE = "5IIR"
GROUPE = "G4"
SITE = "Roudani"

SITES_EMSI = ["Roudani", "Maarif", "Centre", "CFC", "Oranger", "Moulay Youssef"]
assert NOM and PRENOM and CLASSE and GROUPE, "Complétez NOM, PRENOM, CLASSE et GROUPE."
assert SITE in SITES_EMSI, f"SITE doit être l'un de {SITES_EMSI}"
NOM_FICHIER = f"TP{NUM_TP}_{NOM.upper().replace(' ', '-')}_{PRENOM.capitalize()}_{CLASSE}_{GROUPE}_{SITE.replace(' ', '')}.ipynb"
print(f"Étudiant : {PRENOM} {NOM.upper()} | Classe : {CLASSE} | Groupe : {GROUPE} | EMSI {SITE}")
print("➡️ Nom du fichier à rendre :", NOM_FICHIER)


### 0.2 Installation et imports

In [ ]:
# Installation à faire une seule fois dans le noyau Python, si nécessaire :
# python -m pip install scikit-learn nltk pandas numpy
import importlib.util
for module in ("sklearn", "nltk", "pandas", "numpy"):
    if importlib.util.find_spec(module) is None:
        raise ImportError(f"Module manquant : {module}. Consultez la commande ci-dessus.")
print("✅ scikit-learn, NLTK, pandas et numpy disponibles")


In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # évite le plantage du noyau (conflit OpenMP numpy / scikit-learn sous Windows-Anaconda)
import warnings
warnings.filterwarnings("ignore")
import re, html, math
from collections import Counter
import numpy as np
import pandas as pd
import nltk
try:
    nltk.data.find("corpora/stopwords")
except LookupError:
    nltk.download("stopwords", quiet=True, raise_on_error=True)
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
pd.set_option("display.max_colwidth", 90)

def verifier(description, test):
    """Affiche ✅ si test() est vrai, ❌ sinon (sans interrompre le notebook)."""
    try:
        ok = bool(test())
    except Exception as e:
        print(f"❌ {description} — {type(e).__name__} : {e}")
        return
    print(("✅ " if ok else "❌ ") + description)

print("Imports réussis")


### 0.3 Le corpus et la chaîne de prétraitement (fournis)
`CORPUS` : les 100 avis du fil rouge. `preprocess()` : la fonction du **TP2** (texte → tokens). `analyseur()` la complète comme au **TP3** : on retire la ponctuation et les **mots vides adaptés** (la négation est gardée, les mots du domaine sont retirés). On donnera `analyseur` à scikit-learn.

In [ ]:
AVIS = [
    ("Le tajine était délicieux et le service très rapide. Je recommande !", "pos"),
    ("Service TRÈS lent, le plat est arrivé froid. Je ne recommande pas.", "neg"),
    ("Les serveurs sont aimables et le cadre est magnifique 😍", "pos"),
    ("Le couscous n'était pas bon du tout, trop salé.", "neg"),
    ("Trooop bien !!! Livraison rapide et repas encore chaud.", "pos"),
    ("Livraison en retard de 45 minutes, la commande était incomplète.", "neg"),
    ("Très bon rapport qualité-prix, les portions sont généreuses.", "pos"),
    ("Le restaurant est sale et le personnel n'est pas aimable.", "neg"),
    ("J'ai adoré le thé à la menthe et les pâtisseries marocaines.", "pos"),
    ("Les prix sont trop élevés pour des plats sans goût.", "neg"),
    ("Accueil chaleureux, on s'est régalés en famille. Merci @dar_tajine !", "pos"),
    ("Plus jamais ! Le poisson n'était pas frais.", "neg"),
    ("Cadre agréable, musique douce, serveurs souriants.", "pos"),
    ("Commande annulée sans explication, service client injoignable.", "neg"),
    ("Le meilleur restaurant de Rabat, les brochettes sont excellentes.", "pos"),
    ("Les frites étaient froides et la sauce n'avait aucun goût.", "neg"),
    ("tres bon resto, je reviendrai avec mes amis", "pos"),
    ("Pas bon, pas chaud, pas rapide : rien ne va.", "neg"),
    ("Réservation facile sur https://dar-tajine.ma et accueil parfait.", "pos"),
    ("Le service était lent mais les plats étaient bons.", "pos"),
    ("Je ne suis pas satisfait, le livreur était impoli.", "neg"),
    ("Une expérience inoubliable, tout était parfait !", "pos"),
    ("Le restaurent est magnifque, les serveurs très gentils.", "pos"),
    ("Les plats sont arrivés froids, je ne commanderai plus jamais.", "neg"),
    ("Le plat est bon.", "pos"),
    ("Le plat n'est pas bon.", "neg"),
]

AVIS_COURS = [
    ("Service rapide et personnel souriant, je recommande ce restaurant.", "pos"),
    ("Le tajine au poulet était délicieux, service rapide.", "pos"),
    ("Très bon rapport qualité prix, le couscous était parfait.", "pos"),
    ("Service lent, le plat était froid. Je ne recommande pas.", "neg"),
    ("Le tajine au poulet est excellent et pas cher.", "pos"),
    ("Très bon accueil, service rapide, je recommande.", "pos"),
    ("Le couscous était froid et le service lent.", "neg"),
    ("Rapport qualité prix correct, mais le service est lent.", "neg"),
    ("Personnel souriant, tajine au poulet délicieux, très bon moment.", "pos"),
    ("Pas cher, très bon, je recommande ce restaurant à Rabat.", "pos"),
]

AVIS_NOUVEAUX = [
    ("Le tajine au poulet et aux olives était parfait, service rapide et souriant.", "pos"),
    ("Attente interminable, service lent et serveur désagréable. Je ne recommande pas.", "neg"),
    ("Très bon couscous royal, la viande était tendre et bien cuite.", "pos"),
    ("La pastilla au poulet était froide et beaucoup trop sucrée.", "neg"),
    ("Excellent rapport qualité prix, je recommande ce restaurant les yeux fermés.", "pos"),
    ("Le thé à la menthe était délicieux, accueil chaleureux comme à la maison.", "pos"),
    ("Livraison rapide mais la pizza était froide et sans goût.", "neg"),
    ("Personnel souriant, cadre agréable et terrasse magnifique avec vue sur la mer.", "pos"),
    ("Prix trop élevés pour des portions minuscules, plus jamais !", "neg"),
    ("Le service était rapide et les plats bien présentés, très bon moment en famille.", "pos"),
    ("Nous avons attendu une heure, la commande était fausse et personne ne s'est excusé.", "neg"),
    ("Les brochettes de kefta étaient excellentes et pas cher du tout.", "pos"),
    ("Le poisson n'était pas frais et les frites étaient trop grasses.", "neg"),
    ("Très bon accueil, service rapide, le tajine de kefta était délicieux.", "pos"),
    ("Restaurant sale, toilettes dégoûtantes, je ne reviendrai jamais.", "neg"),
    ("La harira était excellente, servie bien chaude avec des dattes.", "pos"),
    ("Le service est lent et le serveur a oublié la moitié de la commande.", "neg"),
    ("Cadre magnifique et musique andalouse, une très belle soirée à Fès.", "pos"),
    ("Les plats étaient froids et le pain était sec. Très déçu.", "neg"),
    ("Très bon rapport qualité prix, les salades marocaines sont fraîches et variées.", "pos"),
    ("Livraison en retard, le livreur était impoli et le repas froid.", "neg"),
    ("Le meilleur couscous de Casablanca, je recommande vivement !", "pos"),
    ("Pas cher mais pas bon : la viande était dure et trop salée.", "neg"),
    ("Accueil chaleureux, personnel souriant et plats délicieux, bravo à toute l'équipe.", "pos"),
    ("Le service était correct mais les plats n'avaient aucun goût.", "neg"),
    ("Jus d'orange frais et crêpes marocaines délicieuses au petit déjeuner.", "pos"),
    ("Commande annulée deux fois, service client injoignable, plus jamais !", "neg"),
    ("Le tajine au poulet était délicieux et le service très rapide.", "pos"),
    ("Musique trop forte, service lent et addition trop salée.", "neg"),
    ("Les serveurs sont très aimables et le cadre est très agréable.", "pos"),
    ("La viande n'était pas cuite et le serveur ne nous a même pas écoutés.", "neg"),
    ("Très bon restaurant, les desserts sont délicieux et pas cher.", "pos"),
    ("Attente trop longue pour une table, puis service lent. Dommage.", "neg"),
    ("Une pastilla au poisson délicieuse et un thé à la menthe parfait.", "pos"),
    ("Les frites étaient froides et le burger sans goût, je ne recommande pas.", "neg"),
    ("Terrasse agréable, service rapide et prix raisonnables à Rabat.", "pos"),
    ("Le couscous était froid et la sauce trop salée, très déçu.", "neg"),
    ("Personnel souriant et efficace, tout était parfait, merci !", "pos"),
    ("Hygiène douteuse, la cuisine était sale et les verres mal lavés.", "neg"),
    ("tres bon tajine, service rapide, je recommande 👍", "pos"),
    ("Le serveur était désagréable et la commande est arrivée en retard.", "neg"),
    ("Très bon moment entre amis, les grillades étaient délicieuses.", "pos"),
    ("Rapport qualité prix décevant, les portions sont trop petites.", "neg"),
    ("Accueil chaleureux et service rapide, le tajine au poulet est un régal.", "pos"),
    ("Le plat est arrivé froid et le service était lent, quelle déception.", "neg"),
    ("Livraison rapide et repas encore chaud, je recommande ce restaurant.", "pos"),
    ("Pas de climatisation, salle bruyante et service lent.", "neg"),
    ("La pastilla au poulet est la meilleure de Marrakech, très bon accueil.", "pos"),
    ("Les prix ont augmenté mais la qualité a baissé, plus jamais.", "neg"),
    ("Délicieux couscous aux sept légumes, pas cher et copieux.", "pos"),
    ("Le thé était froid et les pâtisseries pas fraîches, déçu.", "neg"),
    ("Service impeccable, plats délicieux et cadre magnifique, bravo !", "pos"),
    ("Réservation perdue, attente d'une heure, je ne recommande pas ce restaurant.", "neg"),
    ("Excellent accueil, le chef est venu nous saluer, très bon souvenir.", "pos"),
    ("Le tajine de poisson manquait de sel et les légumes étaient trop cuits.", "neg"),
    ("Rapport qualité prix imbattable, très bon restaurant familial.", "pos"),
    ("Service lent, plats froids et serveurs débordés. Dommage !!!", "neg"),
    ("Jus frais, salades variées et service rapide, très bon petit restaurant.", "pos"),
    ("La commande était incomplète et la livraison en retard d'une heure.", "neg"),
    ("Les brochettes étaient délicieuses et le personnel très souriant.", "pos"),
    ("Je ne recommande pas : viande dure, frites froides, prix élevés.", "neg"),
    ("Ambiance familiale, accueil chaleureux et tajine au poulet délicieux.", "pos"),
    ("Le service était rapide mais le couscous n'avait aucun goût.", "neg"),
    ("Très bon rapport qualité prix et service rapide, je recommande !", "pos"),
]

CORPUS = AVIS + AVIS_COURS + AVIS_NOUVEAUX          # le corpus fil rouge complet
textes = [texte for texte, _ in CORPUS]
etiquettes = [etiquette for _, etiquette in CORPUS]
print(len(AVIS), "avis du TP3 +", len(AVIS_COURS), "avis du cours +", len(AVIS_NOUVEAUX), "nouveaux avis =", len(CORPUS), "avis")
print(Counter(etiquettes))

# ---- chaîne du TP2 (corrigé) ----
def espaces(t):
    return re.sub(r"\s+", " ", t).strip()

def retirer_html(t):
    return espaces(re.sub(r"<[^>]+>", " ", html.unescape(t)))

JETONS = [
    (r"https?://\S+|www\.\S+", " <URL> "),
    (r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+", " <EMAIL> "),
    (r"(?<!\w)@\w+", " <USER> "),
    (r"#(\w+)", r" \1 "),
    (r"(?<!\d)(?:0|\+212 ?)[67](?:[ -]?\d{2}){4}(?!\d)", " <TEL> "),
]
def remplacer_jetons(t):
    for motif, remplacement in JETONS:
        t = re.sub(motif, remplacement, t)
    return espaces(t)

def reduire_repetitions(t):
    t = re.sub(r"([^\W\d_])\1{2,}", r"\1\1", t)
    return re.sub(r"([!?.])\1+", r"\1", t)

EMOJI = r"[\U0001F300-\U0001FAFF☀-➿]"
EMOJIS_POS = {"😍", "👍", "😋", "🎉", "🎂"}
EMOJIS_NEG = {"😡", "👎", "😞", "😒", "☹"}
def jeton_emoji(m):
    e = m.group()
    return " <POS> " if e in EMOJIS_POS else " <NEG> " if e in EMOJIS_NEG else " <EMOJI> "

def traiter_emojis(t, mode="jetons"):
    if mode == "garder":
        return t
    if mode == "retirer":
        return espaces(re.sub(EMOJI, " ", t))
    return espaces(re.sub(EMOJI, jeton_emoji, t))

def nettoyer(t, emojis="jetons"):
    t = retirer_html(t)
    t = remplacer_jetons(t)
    t = reduire_repetitions(t)
    return espaces(traiter_emojis(t, emojis))

JETON = r"<[A-Z]+>"
PARTIES = [r"[Aa]ujourd['’]hui", JETON, r"(?:[A-Z]\.){2,}", r"(?:M|Mme|Dr|etc)\.",
           r"\d+(?:[.,]\d+)?(?![^\W\d_])", r"[^\W\d_]+['’]", r"\w+(?:-\w+)*", r"[^\w\s]"]
TOKEN = re.compile("|".join(PARTIES))
def tok_regex(t):
    return TOKEN.findall(t)

def preprocess(texte, emojis="jetons", minuscules=True):
    """Texte brut → liste de tokens propres (livrable du TP2)."""
    tokens = tok_regex(nettoyer(texte, emojis))
    if minuscules:
        tokens = [t if re.fullmatch(JETON, t) else t.lower() for t in tokens]
    return tokens

NEGATION = {"ne", "n", "pas", "plus", "jamais", "rien", "aucun", "sans", "ni"}
MOTS_VIDES = (set(stopwords.words("french")) - NEGATION) | {"restaurant", "resto", "plat", "plats"}

def analyseur(texte):
    """preprocess() du TP2, sans ponctuation ni mots vides adaptés (TP3)."""
    return [t for t in preprocess(texte)
            if re.search(r"\w", t) and t.rstrip("'’") not in MOTS_VIDES]

print(textes[1]); print(" ->", analyseur(textes[1]))
print(textes[18]); print(" ->", analyseur(textes[18]))

---
## Partie 1 — Le sac de mots (0:15 – 0:30)
### Exercice 1.1 ★ CountVectorizer par défaut
Vectorisez `textes` avec un `CountVectorizer()` **par défaut**. Calculez la **densité** de la matrice : la part des cases non nulles (`X.nnz` divisé par le nombre total de cases).

In [ ]:
vec_brut = CountVectorizer()
X_brut = vec_brut.fit_transform(textes)
densite = X_brut.nnz / np.prod(X_brut.shape)
print("matrice :", X_brut.shape, f"| densité : {densite:.1%}")
print(vec_brut.get_feature_names_out()[:15])


In [ ]:
# ✅ Vérification (ne pas modifier)

verifier("100 lignes", lambda: X_brut.shape[0] == 100)
verifier("densité", lambda: abs(densite - X_brut.nnz / np.prod(X_brut.shape)) < 1e-12)

### Exercice 1.2 ★★ Avec notre prétraitement
Créez `vec_bow = CountVectorizer(analyzer=analyseur)` et la matrice `X_bow`. Puis calculez `totaux` : le nombre d'occurrences de chaque mot dans **tout** le corpus (somme des colonnes : `X_bow.sum(axis=0)`, transformée en tableau 1D avec `np.asarray(...).ravel()`).

In [ ]:
vec_bow = CountVectorizer(analyzer=analyseur)
X_bow = vec_bow.fit_transform(textes)
mots_bow = vec_bow.get_feature_names_out()
totaux = np.asarray(X_bow.sum(axis=0)).ravel()
print("vocabulaire : par défaut", X_brut.shape[1], "mots | avec analyseur", X_bow.shape[1], "mots")
print([(mots_bow[j], int(totaux[j])) for j in np.argsort(totaux)[::-1][:10]])


In [ ]:
# ✅ Vérification (ne pas modifier)

verifier("X_bow utilise l'analyseur", lambda: X_bow.shape == (100, len(set(t for x in textes for t in analyseur(x)))))
verifier("totaux", lambda: len(totaux) == X_bow.shape[1] and totaux.sum() == X_bow.sum())

#### 📝 Interprétation 1
(a) Quelle est la densité de la matrice ? Que signifie « matrice creuse » ici ?
(b) Comparez les deux vocabulaires : quels mots le `CountVectorizer` par défaut garde-t-il que l'analyseur retire, et inversement (pensez à « n' », aux jetons) ?

**Réponse 1.**

(a) La matrice brute a **100 lignes × 252 colonnes** et une densité de **3,8 %** : environ 96,2 % de ses cases sont nulles. Elle est dite *creuse* parce que chaque avis ne contient qu'une petite partie des 252 mots du vocabulaire ; la représentation sparse ne stocke que les cases non nulles.

(b) Le vocabulaire passe de **252 mots** par défaut à **218** avec notre analyseur. Par défaut, `CountVectorizer` garde notamment les mots vides (`le`, `et`, `de`), les mots de domaine (`restaurant`, `plat`) et découpe l'URL en éléments comme `https` ; notre analyseur les retire ou remplace l'URL par `<URL>`. À l'inverse, il conserve `n'` (la négation), `<POS>`, `<USER>`, `<URL>` et les formes avec trait d'union comme `qualité-prix`. Le découpage par défaut ignore les tokens d'un seul caractère, dont `n`.


---
## Partie 2 — TF-IDF (0:30 – 0:50)
### Exercice 2.1 ★★ Vérifier le calcul à la main du cours
Complétez `idf_cours(docs_tok)` : le dictionnaire `{mot: log₁₀(N / df)}` (N = nombre de documents, df = nombre de documents qui contiennent le mot). Vérifiez les valeurs du cours : 0 ; ≈ 0,176 ; ≈ 0,477.

In [ ]:
docs_cours = [["tajine", "délicieux", "service", "rapide"],
              ["tajine", "froid", "service", "lent"],
              ["service", "rapide", "service", "souriant"]]

def idf_cours(docs_tok):
    N = len(docs_tok)
    vocab = {m for d in docs_tok for m in d}
    return {m: math.log10(N / sum(m in d for d in docs_tok)) for m in vocab}

idf = idf_cours(docs_cours)
print({m: round(v, 3) for m, v in sorted(idf.items())})


In [ ]:
# ✅ Vérification (ne pas modifier)

verifier("service : 0", lambda: abs(idf["service"]) < 1e-12)
verifier("rapide : 0,176 ; souriant : 0,477", lambda: round(idf["rapide"], 3) == 0.176 and round(idf["souriant"], 3) == 0.477)

### Exercice 2.2 ★ TfidfVectorizer sur le corpus
Créez `vec_tfidf = TfidfVectorizer(analyzer=analyseur)` et `X_tfidf`. La cellule affiche l'IDF (formule lissée de scikit-learn) de quelques mots.

In [ ]:
vec_tfidf = TfidfVectorizer(analyzer=analyseur)
X_tfidf = vec_tfidf.fit_transform(textes)
mots = vec_tfidf.get_feature_names_out()
idf_sk = dict(zip(mots, vec_tfidf.idf_))
for m in ["service", "rapide", "pas", "délicieux", "kefta", "climatisation"]:
    print(f"{m:14} df = {int(np.sum(X_tfidf[:, vec_tfidf.vocabulary_[m]].toarray() > 0)):2}   idf = {idf_sk[m]:.2f}")


In [ ]:
# ✅ Vérification (ne pas modifier)

verifier("X_tfidf", lambda: X_tfidf.shape == X_bow.shape)
verifier("lignes normalisées (longueur 1)", lambda: np.allclose(np.linalg.norm(X_tfidf.toarray(), axis=1), 1))

### Exercice 2.3 ★★ Les mots caractéristiques d'un avis
Complétez `mots_caracteristiques(i, k)` : les k mots de plus grand TF-IDF de l'avis i, avec leur score arrondi à 2 décimales (`np.argsort`).

In [ ]:
def mots_caracteristiques(i, k=3):
    ligne = X_tfidf[i].toarray()[0]
    meilleurs = np.argsort(ligne)[::-1][:k]
    return [(mots[j], round(float(ligne[j]), 2)) for j in meilleurs]

for i in (0, 3, 40, 63):
    print(i, textes[i]); print("   ->", mots_caracteristiques(i))


In [ ]:
# ✅ Vérification (ne pas modifier)

verifier("k mots, du plus grand score au plus petit", lambda: [s for _, s in mots_caracteristiques(0, 4)] == sorted([s for _, s in mots_caracteristiques(0, 4)], reverse=True))

#### 📝 Interprétation 2
(a) Comparez l'IDF de « service » et de « kefta » : pourquoi une telle différence ? Pourquoi « service » ne vaut-il pas 0 ici, contrairement au calcul du cours ?
(b) Les mots caractéristiques des avis 0, 3, 40 et 63 résument-ils bien ces avis ?

**Réponse 2.**

(a) `service` apparaît dans **30 avis** et son IDF scikit-learn est **2,18** ; `kefta` n'apparaît que dans **2 avis** et son IDF est **4,52**. Un mot rare distingue davantage les documents. Dans le calcul du cours, `service` apparaît dans les **3 documents sur 3** : `log₁₀(3/3) = 0`. Ici, il n'apparaît que dans 30 avis sur 100 ; scikit-learn utilise en plus l'IDF lissé `ln((1+N)/(1+df)) + 1`, donc sa valeur n'est pas nulle.

(b) Les mots de l'avis **0** (`délicieux`, `tajine`, `recommande`) et de l'avis **63** (`poulet`, `délicieux`, `tajine`) résument bien les plats appréciés. Dans l'avis **3**, `salé` et `couscous` sont informatifs, mais `tout` est remonté à cause de « pas bon du tout » : le mot seul ne résume pas le jugement. Dans l'avis **40**, `yeux` et `fermés` dominent parce que l'expression « les yeux fermés » est rare ; séparés, ces mots cachent le sens de la recommandation. TF-IDF mesure la singularité des termes, pas leur interprétation complète.


---
## Partie 3 — Similarité cosinus (0:50 – 1:15)
### Exercice 3.1 ★ Le cosinus à la main
Complétez `cosinus(u, v)` pour deux vecteurs numpy : `u @ v / (‖u‖ × ‖v‖)` (`np.linalg.norm`) ; renvoyez 0 si l'un des deux est nul.

In [ ]:
def cosinus(u, v):
    nu, nv = np.linalg.norm(u), np.linalg.norm(v)
    return float(u @ v / (nu * nv)) if nu and nv else 0.0

print(round(cosinus(np.array([1, 1, 0]), np.array([1, 0, 1])), 3))
u, v = X_bow[0].toarray()[0], X_bow[27].toarray()[0]
print(round(cosinus(u, v), 3), "| scikit-learn :", round(cosine_similarity(X_bow[0], X_bow[27])[0, 0], 3))


In [ ]:
# ✅ Vérification (ne pas modifier)

verifier("cos((1,1,0), (1,0,1)) = 0,5", lambda: abs(cosinus(np.array([1, 1, 0]), np.array([1, 0, 1])) - 0.5) < 1e-12)
verifier("vecteur nul", lambda: cosinus(np.zeros(3), np.ones(3)) == 0)
verifier("comme scikit-learn", lambda: abs(cosinus(u, v) - cosine_similarity(X_bow[0], X_bow[27])[0, 0]) < 1e-9)

### Exercice 3.2 ★★ Les deux avis les plus proches
`S = cosine_similarity(X_tfidf)` donne la matrice 100 × 100 des cosinus. Mettez la diagonale à 0 (un avis est identique à lui-même), puis trouvez la case maximale (`np.argmax` et `np.unravel_index`).

In [ ]:
S = cosine_similarity(X_tfidf)
S2 = S.copy()
np.fill_diagonal(S2, 0)
i, j = np.unravel_index(np.argmax(S2), S2.shape)
print(f"cosinus = {S2[i, j]:.2f}")
print(i, textes[i])
print(j, textes[j])


In [ ]:
# ✅ Vérification (ne pas modifier)

verifier("matrice 100 × 100", lambda: S.shape == (100, 100) and np.allclose(np.diag(S), 1))
verifier("le maximum hors diagonale", lambda: i != j and np.isclose(S2[i, j], S2.max()))

### Exercice 3.3 ★★ Les voisins d'un avis
Complétez `voisins(i, k)` : les k avis les plus proches de l'avis i (sans lui-même), sous forme de couples (score arrondi, numéro).

In [ ]:
def voisins(i, k=3):
    scores = S[i].copy()
    scores[i] = -1
    ordre = np.argsort(scores)[::-1][:k]
    return [(round(float(scores[j]), 2), int(j)) for j in ordre]

print(textes[1])
for s, j in voisins(1):
    print("  ", s, textes[j])


In [ ]:
# ✅ Vérification (ne pas modifier)

verifier("k voisins, sans l'avis lui-même", lambda: len(voisins(1, 5)) == 5 and 1 not in [j for _, j in voisins(1, 5)])

#### 📝 Interprétation 3
Les deux avis les plus proches et les voisins de l'avis 1 vous semblent-ils pertinents ? Sur quels mots repose la similarité ? Citez un voisin moins pertinent et expliquez pourquoi.

**Votre réponse :**

_…_

---
## Partie 4 — Le mini moteur de recherche (1:15 – 1:40) · livrable
Complétez `search(query, k)` :
1. vectoriser la requête avec le **même** vectoriseur : `vec_tfidf.transform([query])` ;
2. calculer le cosinus entre la requête et tous les avis ;
3. renvoyer un tableau des k meilleurs avis (colonnes `score`, `avis`, `étiquette`), du plus proche au moins proche, **sans** les avis de score nul.

In [ ]:
def search(query, k=5):
    q = ...          # ← À COMPLÉTER
    scores = ...     # ← À COMPLÉTER : cosine_similarity(q, X_tfidf)[0]
    ordre = [j for j in np.argsort(scores)[::-1][:k] if scores[j] > 0]
    return pd.DataFrame({"score": [round(float(scores[j]), 3) for j in ordre],
                         "avis": [textes[j] for j in ordre],
                         "étiquette": [etiquettes[j] for j in ordre]})

search("service lent")

In [ ]:
# ✅ Vérification (ne pas modifier)

_r = search("service lent")
verifier("colonnes", lambda: list(_r.columns) == ["score", "avis", "étiquette"])
verifier("scores décroissants", lambda: list(_r["score"]) == sorted(_r["score"], reverse=True) and len(_r) == 5)
verifier("le premier avis contient 'lent'", lambda: "lent" in _r.loc[0, "avis"].lower())
verifier("requête sans mot connu : tableau vide", lambda: len(search("zzz")) == 0)

### Les 6 requêtes de test (fourni)

In [ ]:
REQUETES = ["tajine au poulet délicieux", "livraison en retard", "pas cher", "accueil chaleureux à Rabat",
            "restaurent magnifque", "service vite"]
for r in REQUETES:
    res = search(r, 3)
    print(f"\n🔎 {r}  ({len(res)} résultat(s))")
    for _, ligne in res.iterrows():
        print(f"   {ligne['score']:.2f} [{ligne['étiquette']}] {ligne['avis']}")

---
## 🚀 Pour aller plus loin
### A. ★★ Bigrammes
Créez `vec_bi = TfidfVectorizer(analyzer=analyseur_bi)` où `analyseur_bi` renvoie les mots **et** les bigrammes (« pas cher »). Refaites la requête « pas cher ».

In [ ]:
def analyseur_bi(texte):
    m = analyseur(texte)
    return m + ...   # ← À COMPLÉTER : les bigrammes " ".join(...) de m

vec_bi = TfidfVectorizer(analyzer=analyseur_bi)
X_bi = vec_bi.fit_transform(textes)
s = cosine_similarity(vec_bi.transform(["pas cher"]), X_bi)[0]
for j in np.argsort(s)[::-1][:3]:
    print(round(float(s[j]), 2), etiquettes[j], textes[j])

---
## ✔️ Avant de rendre
- [ ] Identification ; `Noyau → Redémarrer et tout exécuter` sans erreur ; vérifications des parties 1 à 4 en ✅
- [ ] `search(query)` testée sur les 6 requêtes ; P@3 calculées
- [ ] Interprétations rédigées, dont l'**analyse du moteur** (partie 5) : c'est le livrable